# EDA - DonneesRH.xlsx
## Sport Data Solution - HR Data Analysis

**Purpose:** Understand the raw HR data structure, quality and content before building the ETL pipeline. \
**File:** `data/raw/DonneesRH.xlsx` \
**Date:** 2026-03-13

## Section 1 - Setup & Imports

In [1]:
import warnings
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_colwidth', 50)
warnings.filterwarnings('ignore')

DATA_PATH = Path("../data/raw/DonneesRH.xlsx")

print("Librairies loaded successfully")
print(f"Data path: {DATA_PATH}")
print(f"File exists: {DATA_PATH.exists()}")

Librairies loaded successfully
Data path: ../data/raw/DonneesRH.xlsx
File exists: True


## Section 2 - Load Data
Loading the raw Excel file into a pandas DataFrame for analysis.

In [2]:
df = pd.read_excel(DATA_PATH, engine="openpyxl")

print("File loaded successfully")
print(f"Shape: {df.shape[0]} rows * {df.shape[1]} columns")

File loaded successfully
Shape: 161 rows * 11 columns


## Section 3 - Sample

Displaying the first 10 rows of raw data to confirm exact format of each column before designing ETL transformations.

In [13]:
print("=== FIRST 10 ROWS ===")
display(df.head(10))

=== FIRST 10 ROWS ===


,ID salarié,Nom,Prénom,Date de naissance,BU,Date d'embauche,Salaire brut,Type de contrat,Nombre de jours de CP,Adresse du domicile,Moyen de déplacement
0,59019,Colin,Audrey,1990-07-06,Marketing,2020-12-14,30940,CDI,29,"128 Rue du Port, 34000 Frontignan",Transports en commun
1,19841,Ledoux,Monique,1962-01-06,R&D,2020-07-07,74360,CDI,26,"68 Rue du Port, 34970 Saint-Clément-de-Rivière",véhicule thermique/électrique
2,56482,Dumont,Michelle,1976-08-09,Ventes,2022-03-29,51390,CDI,27,"100 Av. de la Gare, 30900 Nîmes",véhicule thermique/électrique
3,21886,Toussaint,Judith,1962-09-10,Support,2021-12-12,70320,CDI,29,"53 Av. de la Gare, 34970 Lattes",Marche/running
4,81001,Bailly,Michelle,1975-04-20,Ventes,2025-02-19,46870,CDD,29,"74 Rue des Fleurs, 34970 Lattes",Marche/running
5,17757,Bazin,Margaret,1982-11-24,Support,2023-04-17,31460,CDI,25,"Av. de La Mer, 34470 Pérols",Vélo/Trottinette/Autres
6,17036,Jacques,Julien,1970-12-19,Marketing,2021-08-21,72480,CDI,29,"28 Rue du Port, 34140 Mèze",véhicule thermique/électrique
7,36913,Pons,Brigitte,1977-12-27,Finance,2020-05-26,59790,CDI,28,"169 Rue Jean Jaures, 34730 Prades-le-Lez",Transports en commun
8,79006,Rousset,Jérome,1979-02-01,R&D,2020-09-07,29240,CDI,28,"44 Rue Jean Jaures, 34730 Prades-le-Lez",véhicule thermique/électrique
9,62296,Chauvin,Zakaria,1987-10-02,Finance,2024-11-23,68680,CDD,26,"31 Av. Grassion Cibrand, 34130 Mauguio",Vélo/Trottinette/Autres


## Section 4 - Shape & Column Names
Examining the exact structure and column naming of the dataset.

In [3]:
print("=== SHAPE ===")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

print("\n=== Column Names ===")
for i, col in enumerate(df.columns):
    print(f" [{i}] {col}")

=== SHAPE ===
Rows: 161
Columns: 11

=== Column Names ===
 [0] ID salarié
 [1] Nom
 [2] Prénom
 [3] Date de naissance
 [4] BU
 [5] Date d'embauche
 [6] Salaire brut
 [7] Type de contrat
 [8] Nombre de jours de CP
 [9] Adresse du domicile
 [10] Moyen de déplacement


### Observations

**Issues identified:**
1. Inconsistent casing -> standardize to snake_case
2. Spaces in column names -> replace with underscores
3. Special characters (accents) -> remove for ASCII safety
4. Link words (de, du, d') -> remove, not meaningful
5. Abbreviations (CP) -> replace with meaningful names

**Action required:**
Apply column renaming mapping in load_employees.py
using snake_case convention consistent with database schema.
All Excel columns mapped to Python snake_case → DB rh_ prefix columns.

## Section 5 - Data Types
Examining what pandas inferred as the data type for each column.\
Expected types are compared against inferred types to identify necessary transformations for the ETL piepeline. 

In [4]:
print("=== INFERRED DATA TYPES ===")
print(df.dtypes)

print("\n=== TYPE SUMMARY ===")
type_counts = df.dtypes.value_counts()
for dtype, count in type_counts.items():
    print(f"{dtype}: {count} columns")

=== INFERRED DATA TYPES ===
ID salarié                        int64
Nom                                 str
Prénom                              str
Date de naissance        datetime64[us]
BU                                  str
Date d'embauche          datetime64[us]
Salaire brut                      int64
Type de contrat                     str
Nombre de jours de CP             int64
Adresse du domicile                 str
Moyen de déplacement                str
dtype: object

=== TYPE SUMMARY ===
str: 6 columns
int64: 3 columns
datetime64[us]: 2 columns


### Observations

Overall types are well inferred by pandas. Three points to note:

1. ID salarie: int64 -> must convert to string in ETL
   IDs are identifiers, not numbers. Never used in calculations.

2. Salaire brut: int64 -> compatible with DB NUMERIC(10,2)
   No decimals in source data. No action needed.

3. Adresse du domicile: str -> needs splitting in ETL
   Single string must be split into 4 columns:
   street_number, street_name, postal_code, city.
   Address format to be confirmed in Section 9 sample.

## Section 6 - Missing Values

Identifying columns with missing values, their count percentage.\
Missing values in NOT NULL columns will cause ETL failures and must be handle before loading into the database.

In [5]:
print("=== MISSING VALUES ===")
missing_count = df.isnull().sum()
missing_pct = (missing_count / len(df) *100).round(2)

missing_df = pd.DataFrame({
    'missing_count': missing_count,
    'missing_pct': missing_pct,
    'nullable_in_db': [False, False, False, False, False, False, False, False, False, False, False]
})

missing_df = missing_df.sort_values('missing_count', ascending=False)
print(missing_df)

print("\n=== SUMMARY ===")
total_missing = missing_count.sum()
columns_with_missing = (missing_count > 0).sum()
print(f"Total missing values: {total_missing}")
print(f"Columns with missing data: {columns_with_missing}/{df.shape[1]}")

=== MISSING VALUES ===
                       missing_count  missing_pct  nullable_in_db
ID salarié                         0         0.00           False
Nom                                0         0.00           False
Prénom                             0         0.00           False
Date de naissance                  0         0.00           False
BU                                 0         0.00           False
Date d'embauche                    0         0.00           False
Salaire brut                       0         0.00           False
Type de contrat                    0         0.00           False
Nombre de jours de CP              0         0.00           False
Adresse du domicile                0         0.00           False
Moyen de déplacement               0         0.00           False

=== SUMMARY ===
Total missing values: 0
Columns with missing data: 0/11


### Observation

Dataset is complete, no missing values across all 11 columns.
No missing value handling required in the ETL pipeline.

## Section 7 - Duplicates
Checking for two types of duplicates:
1. Full row duplicates: every column identical across rows
2. ID duplicates: same employee ID with different data

Both types will cause ETL failures due to PRIMARY KEY constraint on rh_employee_id in the empployees table.

In [6]:
print("=== FULL ROW DUPLICATES ===")
full_duplicates = df.duplicated()
print(f"Duplicate rows: {full_duplicates.sum()}")

if full_duplicates.sum() > 0:
    print("\nDuplicate rows content:")
    print(df[full_duplicates])

print("\n=== DUPLICATE EMPLOYEE ID ===")
id_column = 'ID salarié'
id_duplicates = df.duplicated(subset=[id_column])
print(f"Duplicate IDs: {id_duplicates.sum()}")

if id_duplicates.sum() > 0:
    duplicated_ids = df[df.duplicated(subset=[id_column], keep=False)]
    print("\nRows with duplicate IDs:")
    print(duplicated_ids.sort_values(id_column))

print("\n=== SUMMARY ===")
print(f"Total rows: {len(df)}")
print(f"Full rows duplicates: {full_duplicates.sum()}")
print(f"Duplicate IDs: {id_duplicates.sum()}")
print(f"Unique employee IDs: {df[id_column].nunique()}")

=== FULL ROW DUPLICATES ===
Duplicate rows: 0

=== DUPLICATE EMPLOYEE ID ===
Duplicate IDs: 0

=== SUMMARY ===
Total rows: 161
Full rows duplicates: 0
Duplicate IDs: 0
Unique employee IDs: 161


### Observation

No duplicates found, full row or ID level.
161 rows = 161 unique employee IDs confirms data integrity.

Dataset is healthy for this dimension but production ETL
must still include defensive checks:
- UNIQUE constraint on rh_employee_id in DB already handles this
- ETL should catch duplicate ID errors and log them explicitly rather than crashing silently

## Section 8 - Unique values
Analysing unique values for categorical columns.\
Critical for validating ENUM mappings and transport mode values that must match our database ENUM type definitions

In [7]:
CATEGORICAL_THRESHOLD = 15

print("=== UNIQUE VALUES PER COLUMN ===\n")
for col in df.columns:
    n_unique = df[col].nunique()
    print(f"--- {col} ({n_unique} unique values) ---")
    
    if n_unique <= CATEGORICAL_THRESHOLD:
        value_counts = df[col].value_counts()
        for value, count in value_counts.items():
            pct = round(count / len(df) * 100, 2)
            print(f"  {value:<40} count: {count:<5} ({pct}%)")
    else:
        print(f"  High cardinality — {n_unique} unique values, sample:")
        for val in df[col].head(3):
            print(f"  {val}")
    print()

=== UNIQUE VALUES PER COLUMN ===

--- ID salarié (161 unique values) ---
  High cardinality — 161 unique values, sample:
  59019
  19841
  56482

--- Nom (130 unique values) ---
  High cardinality — 130 unique values, sample:
  Colin
  Ledoux
  Dumont

--- Prénom (113 unique values) ---
  High cardinality — 113 unique values, sample:
  Audrey
  Monique
  Michelle

--- Date de naissance (160 unique values) ---
  High cardinality — 160 unique values, sample:
  1990-07-06 00:00:00
  1962-01-06 00:00:00
  1976-08-09 00:00:00

--- BU (5 unique values) ---
  Finance                                  count: 42    (26.09%)
  Support                                  count: 35    (21.74%)
  Ventes                                   count: 33    (20.5%)
  R&D                                      count: 26    (16.15%)
  Marketing                                count: 25    (15.53%)

--- Date d'embauche (154 unique values) ---
  High cardinality — 154 unique values, sample:
  2020-12-14 00:00:00
  20

### Observation

**Transport mode (critical for ETL):**
Only 68/161 employees (42.24%) eligible for prime:
- Vélo/Trottinette/Autres: 54 (33.54%) -> cycling
- Marche/running: 14 (8.70%) -> walking

ENUM mapping confirmed:
- "véhicule thermique/électrique" -> motorized
- "Transports en commun" -> motorized  
- "Vélo/Trottinette/Autres" -> cycling
- "Marche/running" -> walking

Case inconsistency detected → apply .lower().strip() before mapping.

**Address format:**
Pattern: "{number} {street_name}, {postal_code} {city}"
Split strategy: comma separator then space for street number.
Edge cases: abbreviated streets (Av., Bd.), hyphenated city names.

**Dates:**
Format: datetime with 00:00:00 time component.
Action: strip time part, store as DATE in PostgreSQL.

**Contract & BU:**
No unexpected values. CDI dominant (92.55%).
5 balanced BUs, good for dashboard segmentation.

## Section 9 - Statistics
Statistical analysis of numeric, categorical and date columns.\
Outliers and anomalies identified here must be handle in ETL.

In [8]:
print("=== NUMERIC COLUMNS ===")
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
print(df[numeric_cols].describe().round(2))

print("\n=== CATEGORICAL COLUMNS ===")
categorical_cols = df.select_dtypes(include=['object', 'str']).columns
for col in categorical_cols:
    if df[col].nunique() <= CATEGORICAL_THRESHOLD:
        print(f"\n{col}:")
        print(f"Most frequent: {df[col].mode()[0]} ({df[col].value_counts().iloc[0]} occurrences)")
        print(f"Least frequent: {df[col].value_counts().index[-1]} ({df[col].value_counts().iloc[-1]} occurrences)")

print("\n=== DATE COLUMNS ===")
date_cols = df.select_dtypes(include=['datetime64']).columns
for col in date_cols:
    print(f"\n{col}:")
    print(f"Earliest: {df[col].min().date()}")
    print(f"Latest: {df[col].max().date()}")
    print(f"Range: {(df[col].max() - df[col].min()).days} days")

=== NUMERIC COLUMNS ===
       ID salarié  Salaire brut  Nombre de jours de CP
count      161.00        161.00                 161.00
mean     52622.02      50426.27                  26.95
std      24202.83      14779.61                   1.46
min      12497.00      25570.00                  25.00
25%      33386.00      37910.00                  26.00
50%      49804.00      50580.00                  27.00
75%      71657.00      62760.00                  28.00
max      99514.00      74990.00                  29.00

=== CATEGORICAL COLUMNS ===

BU:
Most frequent: Finance (42 occurrences)
Least frequent: Marketing (25 occurrences)

Type de contrat:
Most frequent: CDI (149 occurrences)
Least frequent: CDD (12 occurrences)

Moyen de déplacement:
Most frequent: véhicule thermique/électrique (73 occurrences)
Least frequent: Marche/running (14 occurrences)

=== DATE COLUMNS ===

Date de naissance:
Earliest: 1960-04-24
Latest: 2002-08-04
Range: 15442 days

Date d'embauche:
Earliest: 2020-03-31


### Observation 

All statistical values are realistic and consistent with the company profile.\
No outliers or anomalies requiring ETL intervention.\
Dataset is statistically healthy.

## Section 10 - Deep Address Analysis
Complete validation of all 161 addresses across 7 checks.\
Anomalies are flagged for ETL handling, never silently dropped.

Checks performed:
1. Missing street number
2. Consistent order (street before postal code)
3. Exactly one comma separator
4. Multiple commas detection
5. Irregular space separators
6. Postal code format (exactly 5 digits)
7. Null-like strings (N/A, -, ?, empty)


In [14]:
print("=== ADDRESS FORMAT ANALYSIS ===")
print("Sample addresses to define parsing strategy: \n")
for addr in df['Adresse du domicile'].head(10):
    parts = addr.split(', ')
    print(f"Full: {addr}")
    print(f"Parts: {parts}")
    print()


=== ADDRESS FORMAT ANALYSIS ===
Sample addresses to define parsing strategy: 

Full: 128 Rue du Port, 34000 Frontignan
Parts: ['128 Rue du Port', '34000 Frontignan']

Full: 68 Rue du Port, 34970 Saint-Clément-de-Rivière
Parts: ['68 Rue du Port', '34970 Saint-Clément-de-Rivière']

Full: 100 Av. de la Gare, 30900 Nîmes
Parts: ['100 Av. de la Gare', '30900 Nîmes']

Full: 53 Av. de la Gare, 34970 Lattes
Parts: ['53 Av. de la Gare', '34970 Lattes']

Full: 74 Rue des Fleurs, 34970 Lattes
Parts: ['74 Rue des Fleurs', '34970 Lattes']

Full: Av. de La Mer, 34470 Pérols
Parts: ['Av. de La Mer', '34470 Pérols']

Full: 28 Rue du Port, 34140 Mèze
Parts: ['28 Rue du Port', '34140 Mèze']

Full: 169 Rue Jean Jaures, 34730 Prades-le-Lez
Parts: ['169 Rue Jean Jaures', '34730 Prades-le-Lez']

Full: 44 Rue Jean Jaures, 34730 Prades-le-Lez
Parts: ['44 Rue Jean Jaures', '34730 Prades-le-Lez']

Full: 31 Av. Grassion Cibrand, 34130 Mauguio
Parts: ['31 Av. Grassion Cibrand', '34130 Mauguio']



In [15]:
addresses = df['Adresse du domicile']
issues = []

print("=== CHECK 1: MISSING STREET NUMBER ===")
no_number = []
for idx, addr in addresses.items():
    parts = addr.split(', ')
    if len(parts) >= 2:
        first_token = parts[0].split(' ')[0]
        if not first_token.isdigit():
            no_number.append((idx, addr))

print(f"Addresses without street number: {len(no_number)}")
if no_number:
    for idx, addr in no_number:
        print(f"  Row {idx}: {addr}")
        issues.append({'row': idx, 'address': addr, 'issue': 'missing_street_number'})

print("\n=== CHECK 2: CONSISTENT ORDER ===")
wrong_order = []
for idx, addr in addresses.items():
    parts = addr.split(', ')
    if len(parts) >= 2:
        second_part = parts[1].strip()
        first_token = second_part.split(' ')[0]
        if not first_token.isdigit() or len(first_token) != 5:
            wrong_order.append((idx, addr))

print(f"Addresses with unexpected order: {len(wrong_order)}")
if wrong_order:
    for idx, addr in wrong_order:
        print(f"  Row {idx}: {addr}")
        issues.append({'row': idx, 'address': addr, 'issue': 'wrong_order'})

print("\n=== CHECK 3 & 4: COMMA COUNT ===")
comma_issues = []
for idx, addr in addresses.items():
    comma_count = addr.count(',')
    if comma_count != 1:
        comma_issues.append((idx, addr, comma_count))

print(f"Addresses with comma count != 1: {len(comma_issues)}")
if comma_issues:
    for idx, addr, count in comma_issues:
        print(f"  Row {idx} ({count} commas): {addr}")
        issues.append({'row': idx, 'address': addr, 'issue': f'{count}_commas'})

print("\n=== CHECK 5: IRREGULAR SPACES ===")
space_issues = []
for idx, addr in addresses.items():
    if '  ' in addr or '\t' in addr:
        space_issues.append((idx, addr))

print(f"Addresses with irregular spaces: {len(space_issues)}")
if space_issues:
    for idx, addr in space_issues:
        print(f"  Row {idx}: {repr(addr)}")
        issues.append({'row': idx, 'address': addr, 'issue': 'irregular_spaces'})

print("\n=== CHECK 6: POSTAL CODE FORMAT ===")
postal_issues = []
for idx, addr in addresses.items():
    parts = addr.split(', ')
    if len(parts) >= 2:
        postal_candidate = parts[1].strip().split(' ')[0]
        if not postal_candidate.isdigit() or len(postal_candidate) != 5:
            postal_issues.append((idx, addr, postal_candidate))

print(f"Addresses with invalid postal code: {len(postal_issues)}")
if postal_issues:
    for idx, addr, postal in postal_issues:
        print(f"  Row {idx} (postal: '{postal}'): {addr}")
        issues.append({'row': idx, 'address': addr, 'issue': f'invalid_postal_{postal}'})

print("\n=== CHECK 7: NULL-LIKE STRINGS ===")
null_like = ['n/a', 'na', '-', '?', '', ' ', 'none', 'null', 'inconnu']
null_issues = []
for idx, addr in addresses.items():
    if addr.strip().lower() in null_like:
        null_issues.append((idx, addr))

print(f"Addresses with null-like values: {len(null_issues)}")
if null_issues:
    for idx, addr in null_issues:
        print(f"  Row {idx}: '{addr}'")
        issues.append({'row': idx, 'address': addr, 'issue': 'null_like_value'})

print("\n=== SUMMARY ===")
print(f"Total addresses analyzed:  {len(addresses)}")
print(f"Total issues found:        {len(issues)}")
print(f"Clean addresses:           {len(addresses) - len(set(i['row'] for i in issues))}")

if issues:
    issues_df = pd.DataFrame(issues)
    print(f"\nIssues breakdown:")
    print(issues_df['issue'].value_counts())

=== CHECK 1: MISSING STREET NUMBER ===
Addresses without street number: 16
  Row 5: Av. de La Mer, 34470 Pérols
  Row 22: Av. de la Mer, Montpellier
  Row 35: 199-97 Av. du Pichagret, 34980 Saint-Gély-du-Fesc
  Row 38: Pl. de la Liberté, 34430 Saint-Jean-de-Védas
  Row 51: Pl. Jean Jaurès, 34000 Montpellier
  Row 81: Pl. de la Liberté, 34470 Pérols
  Row 82: Pl. du Marché, 34250 Palavas-les-Flots
  Row 90: Bd des Sports, 34000 Montpellier
  Row 96: Rue des Écoles, 34120 Lézignan-la-Cèbe
  Row 113: Rue du N, 34920 Le Crès
  Row 114: Av. de la Gare, 34540 Balaruc-les-Bains
  Row 115: Rue Rosset, 34000 Montpellier
  Row 122: Pl. de la Liberté, 34470 Pérols
  Row 125: Rue du Ctre, 34160 Saint-Drézéry
  Row 139: Rue du Ctre, 34160 Saint-Drézéry
  Row 140: Rue des Vignes, 34070 Montpellier

=== CHECK 2: CONSISTENT ORDER ===
Addresses with unexpected order: 1
  Row 22: Av. de la Mer, Montpellier

=== CHECK 3 & 4: COMMA COUNT ===
Addresses with comma count != 1: 1
  Row 36 (0 commas): 455 rout

### Observation

Google Maps API handles address splitting entirely.
No custom parsing logic needed for components.
Raw address string sent to Google -> returns validated components.

Fallback for unresolvable addresses:
- Store raw address in rh_street_name
- Set other address fields to NULL
- Set be_declaration_valid = FALSE
- Flag for manual review
- Employee still loaded, not excluded from pipeline

## Section 11 — Final Conclusions

### Dataset Overview
- Source: DonneesRH.xlsx
- Rows: 161 employees
- Columns: 11
- Expected DB table: employees (161 rows after ETL)

---

### Data Quality Summary

| Dimension        | Result        | Action needed |
|------------------|---------------|---------------|
| Missing values   | 0             | None          |
| Full duplicates  | 0             | None          |
| Duplicate IDs    | 0             | None          |
| Address issues   | 17/161 (10.6%)| See below     |
| Type mismatches  | 2             | See below     |

---

### ETL Actions Required

**1. Column renaming**
All columns must be renamed to snake_case following
the rh_ prefix convention defined in our DB schema.
Remove accents, spaces, link words, special characters.

**2. Type conversions**
- ID salarié: int64 -> string (identifier, not a number)
- Date de naissance: datetime64 -> DATE (strip time component)
- Date d embauche: datetime64 -> DATE (strip time component)

**3. Transport mode mapping**
Apply dictionary mapping with .lower().strip() normalization:
- "véhicule thermique/électrique" -> motorized
- "transports en commun"         -> motorized
- "vélo/trottinette/autres"      -> cycling
- "marche/running"               -> walking

**4. Address handling**
Send raw address string to Google Maps API.
Google returns validated split components directly.
No custom parsing logic needed.

Fallback for unresolvable addresses:
- Store raw string in rh_street_name
- Set remaining address fields to NULL
- Set be_declaration_valid = FALSE
- Employee still loaded, flagged for review

**5. PII encryption**
Apply pgp_sym_encrypt() to these columns before insert:
- rh_last_name, rh_first_name, rh_birth_date
- rh_gross_salary
- rh_street_number, rh_street_name, rh_postal_code, rh_city

**6. Defensive checks (production readiness)**
Even though current dataset is clean, ETL must handle:
- Future missing values in NOT NULL columns -> log + skip row
- Future duplicate IDs -> log + skip row (UNIQUE constraint)
- Unknown transport mode values -> log + skip row
- Google Maps API failures -> fallback strategy above

---

### Address Issues Detail

| Row | Address | Issue | ETL Action |
|-----|---------|-------|------------|
| 5   | Av. de La Mer, 34470 Pérols | No street number | Google Maps |
| 22  | Av. de la Mer, Montpellier | No number, no postal | Google Maps + flag |
| 35  | 199-97 Av. du Pichagret... | Non-standard number | Google Maps + flag |
| 36  | 455 route de mendes... | No comma, no caps | Google Maps + flag |
| 113 | Rue du N, 34920 Le Crès | Truncated name | Google Maps + flag |
| ... | 12 other missing numbers | No street number | Google Maps |

---

### Financial Preview (from EDA)

Based on transport mode distribution:
- Eligible for prime (cycling + walking): 68/161 = 42.2%
- Not eligible (motorized): 93/161 = 57.8%

Estimated prime impact (rough calculation):
- Mean salary: 50,426€
- Prime rate: 5%
- Eligible employees: 68
- Estimated total prime cost: 68 × 50,426€ × 5% = ~171,449€/year

Note: actual amount will vary per individual salary.\
Precise calculation done in ETL -> employee_benefits table.

---

### Confidence Level
Dataset is healthy and well-structured.\
161 employees ready for ETL loading.\
17 address edge cases fully documented and handled.\
No blocking issues, ETL implementation can proceed.